In [22]:
def _curriculum_params(global_steps: int) -> float:
    # 每次读取 proxy，而不是在 __init__ 时复制到本地
    max_model_len = 40960
    max_response_ratio = 0.99
    min_curriculum_epoch = 16
    max_curriculum_epoch = 130
    prompt_length = 1024
    response_length = 39936

    def linear_ratio(epoch: int, max_response_ratio: float, max_curriculum_epoch: int, min_curriculum_epoch: int) -> float:
        """
        linear schedule:
        - epoch = 0             → ratio = max_response_ratio
        - epoch >= max_curriculum_epoch    → ratio = 1.0
        - 0 < epoch < end_ep    → ratio = linear between max_response_ratio and 1.0
        """
        return max_response_ratio - max_response_ratio * (min(max(epoch-min_curriculum_epoch,1) / max_curriculum_epoch, 1))
        # return max_response_ratio - max_response_ratio * (min(epoch / max_curriculum_epoch, 1))

    progress_ratio = linear_ratio(global_steps, max_response_ratio, max_curriculum_epoch, min_curriculum_epoch)
    step_prompt_length = int(prompt_length + response_length * progress_ratio)
    response_length =  max_model_len - step_prompt_length

    return progress_ratio, step_prompt_length, response_length
_curriculum_params(1)

(0.9823846153846154, 40256, 704)

In [1]:
import torch
a = torch.randn([2,3])
# a[0,:300]
a[0,-100:]

tensor([-1.5741,  1.2083, -0.8776])

In [16]:
def _curriculum_params(global_steps: int) -> float:
    max_model_len = 40960
    prompt_length = 1024
    response_length = 39936
    max_response_ratio = 0.99
    min_curriculum_epoch = 16
    max_curriculum_epoch = 130
    # 每次读取 proxy，而不是在 __init__ 时复制到本地
    epoch = global_steps

    def linear_ratio(epoch: int, max_response_ratio: float, max_curriculum_epoch: int) -> float:
        """
        linear schedule:
        - epoch = 0             → ratio = max_response_ratio
        - epoch >= max_curriculum_epoch    → ratio = 1.0
        - 0 < epoch < end_ep    → ratio = linear between max_response_ratio and 1.0
        """
        return max_response_ratio - max_response_ratio * (min(epoch / max_curriculum_epoch, 1))

    progress_ratio = linear_ratio(epoch, max_response_ratio, max_curriculum_epoch)
    step_prompt_length = int(prompt_length + response_length * progress_ratio)
    response_length =  max_model_len - step_prompt_length

    return response_length

_curriculum_params(4)

1616